In [ ]:
!pip install ultralytics supervision opencv-python-headless -q

In [ ]:
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from google.colab import files
from IPython.display import display
import os
import zipfile

In [ ]:
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

In [8]:
# Traffic flows TOP to BOTTOM so allowed angle is 270 degrees
# 0=right, 90=up, 180=left, 270=down
ALLOWED_DIRECTION = 90  # flipped: now UP is allowed, so top-to-bottom vehicles get flagged
ANGLE_THRESHOLD = 90       # flag if more than 90 degrees off
MIN_TRACK_FRAMES = 15      # minimum frames before checking direction
VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

In [9]:
def get_movement_angle(positions):
    if len(positions) < 2:
        return None
    x1, y1 = positions[0]
    x2, y2 = positions[-1]
    dx = x2 - x1
    dy = y1 - y2  # invert Y because image Y goes downward
    angle = np.degrees(np.arctan2(dy, dx)) % 360
    return angle

def angle_difference(a1, a2):
    diff = abs(a1 - a2) % 360
    return min(diff, 360 - diff)

def is_wrong_way(movement_angle):
    if movement_angle is None:
        return False
    return angle_difference(movement_angle, ALLOWED_DIRECTION) > ANGLE_THRESHOLD

In [ ]:
model = YOLO("yolov8m.pt")
tracker = sv.ByteTrack()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

output_path = "output_wrongway.mp4"
violations_dir = "wrongway_violations"
os.makedirs(violations_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

track_history = {}
logged_violations = set()
violation_log = []
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    mask = [int(cls) in VEHICLE_CLASSES for cls in detections.class_id]
    detections = detections[mask]
    detections = tracker.update_with_detections(detections)

    wrong_way_ids = set()

    if detections.tracker_id is not None:
        for i, tracker_id in enumerate(detections.tracker_id):
            bbox = detections.xyxy[i]
            cx = int((bbox[0] + bbox[2]) / 2)
            cy = int((bbox[1] + bbox[3]) / 2)

            if tracker_id not in track_history:
                track_history[tracker_id] = []
            track_history[tracker_id].append((cx, cy))
            track_history[tracker_id] = track_history[tracker_id][-30:]

            positions = track_history[tracker_id]
            if len(positions) < MIN_TRACK_FRAMES:
                continue

            angle = get_movement_angle(positions)

            if is_wrong_way(angle):
                wrong_way_ids.add(tracker_id)
                if tracker_id not in logged_violations:
                    logged_violations.add(tracker_id)
                    snapshot_path = f"{violations_dir}/wrongway_{tracker_id}_{frame_count}.jpg"
                    cv2.imwrite(snapshot_path, frame)
                    violation_log.append({
                        "frame": frame_count,
                        "tracker_id": int(tracker_id),
                        "movement_angle": round(angle, 2),
                        "snapshot": snapshot_path
                    })
                    print(f"WRONG WAY: Vehicle {tracker_id} at frame {frame_count} | angle: {angle:.1f}")

        # Draw tracks
        for tid, positions in track_history.items():
            if len(positions) > 1:
                color = (0, 0, 255) if tid in wrong_way_ids else (0, 255, 0)
                for j in range(1, len(positions)):
                    cv2.line(frame, positions[j-1], positions[j], color, 2)

        labels = [
            f"ID{tid} {VEHICLE_CLASSES.get(int(cls),'v')} {conf:.2f}{'  WRONG WAY!' if tid in wrong_way_ids else ''}"
            for tid, cls, conf in zip(detections.tracker_id, detections.class_id, detections.confidence)
        ]
        frame = box_annotator.annotate(scene=frame, detections=detections)
        frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

    out.write(frame)
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print(f"\nDone. Total wrong-way violations: {len(violation_log)}")

In [11]:
with zipfile.ZipFile("wrongway_violations.zip", "w") as zf:
    for f in os.listdir(violations_dir):
        zf.write(os.path.join(violations_dir, f), f)

files.download(output_path)
files.download("wrongway_violations.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>